# Moshi Compression — Session S0 (Smoke Test)

**Goal:** Confirm the full compute graph fits on dual Kaggle T4s before writing a single real training line.

| Cell | What it does |
|---|---|
| 1 | Disable `torch.compile` globally (no deps needed) |
| 3b | `CUDAGraphed` no-op patch — runs after moshi install, before any `lm_gen` import |
| 2 | Environment verification (dual T4, sm_75, bf16=False) |
| 3 | Pinned installs (transformers 4.44.2, bitsandbytes 0.43.3, moshi editable) |
| 4 | Load teacher + Mimi in fp16 |
| 5 | Shard teacher to cuda:1 |
| 6 | Pre-implementation checklist |
| 7 | Write `smol_temporal.py` to disk |
| 8 | Build student shell via `replace_temporal_transformer` |
| 9 | Integration sanity checks |
| 10 | 10 fake training steps (backward without OOM) |
| 11 | Save checkpoint + MANIFEST.md + env.txt |
| 12 | Push to Kaggle Dataset `moshi-compression-smoke` |

> **Hard rule:** do not start Phase 0 teacher caching (Session S1) until Cell 11 completes without error.


## Cell 1 — Global patches
> `torch.compile` is disabled here (no deps needed).  
> The `CUDAGraphed` no-op patch is applied in **Cell 3b**, immediately after moshi is installed.

In [1]:
# Disable torch.compile globally.
# Kaggle T4 Inductor occasionally emits bf16 intrinsics → "no kernel image" crash.
import os
import torch

os.environ["TORCH_COMPILE_DISABLE"] = "1"
torch._dynamo.config.disable = True
print("torch.compile disabled globally")
print("NOTE: CUDAGraphed patch will be applied in Cell 3b, after moshi is installed.")

torch.compile disabled globally
NOTE: CUDAGraphed patch will be applied in Cell 3b, after moshi is installed.


## Cell 2 — Environment verification

In [2]:
import sys
print("python :", sys.version)
print("torch  :", torch.__version__, "  cuda:", torch.version.cuda)
print("cuda available  :", torch.cuda.is_available())
print("device count    :", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} = {p.name}, sm {p.major}.{p.minor}, "
          f"total {p.total_memory / 1e9:.1f} GB")

# ── Hard assertions ──
assert torch.cuda.device_count() == 2, (
    f"Need dual-GPU Kaggle runtime, got {torch.cuda.device_count()} GPU(s)")

for i in range(2):
    p = torch.cuda.get_device_properties(i)
    assert (p.major, p.minor) == (7, 5), (
        f"Expected T4 (sm_75) on cuda:{i}, got sm_{p.major}.{p.minor}")
    assert p.total_memory >= 15_000_000_000, (
        f"cuda:{i} reports only {p.total_memory/1e9:.1f} GB, expected ~16 GB")

# is_bf16_supported() checks CUDA software version, not silicon.
# On Kaggle it can return True even on T4 (sm_75). The real test is CC >= 8.0.
def hw_bf16_supported():
    return all(
        torch.cuda.get_device_capability(i)[0] >= 8
        for i in range(torch.cuda.device_count())
    )

print("bf16 supported (hardware CC>=8.0):", hw_bf16_supported())  # must be False on T4
assert not hw_bf16_supported(), (
    "Hardware bf16 detected (CC >= 8.0) — are you on a non-T4 GPU? "
    f"Capabilities: {[torch.cuda.get_device_capability(i) for i in range(torch.cuda.device_count())]}"
)

try:
    import flash_attn  # noqa
    print("WARNING: flash_attn imported — unexpected on T4")
except ImportError:
    print("flash_attn not present — expected on T4")

# Anchor cuda:0 so Kaggle GPU ordering stays stable.
torch.cuda.set_device(0)
print("cuda:0 anchored")
print("=== environment check PASSED ===")


python : 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
torch  : 2.9.1+cu128   cuda: 12.8
cuda available  : True
device count    : 2
  cuda:0 = Tesla T4, sm 7.5, total 15.6 GB
  cuda:1 = Tesla T4, sm 7.5, total 15.6 GB
bf16 supported (hardware CC>=8.0): False
flash_attn not present — expected on T4
cuda:0 anchored
=== environment check PASSED ===


## Cell 3 — Pinned installs
> ~2 min. Versions are pinned and non-negotiable.

In [ ]:
import subprocess, sys, os

# PyPI packages — pinned versions, non-negotiable.
for pkg in [
    "transformers==4.44.2",
    "accelerate==0.33.0",
    "bitsandbytes>=0.45,<0.50",  # moshi 0.2.13 requires >=0.45; 0.49.x is fine
    "sentencepiece",
    "einops",
]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# ── Moshi — copy to /kaggle/working first, then install ──────────────────────
# /kaggle/input is read-only. `pip install -e` must write __editable__ metadata
# into the source tree, so it will always fail from /kaggle/input directly.
# We copy the repo to /kaggle/working (writable) once per session.

MOSHI_SRC = "/kaggle/input/datasets/tasfiatanha/moshi-repo/moshi/moshi"
MOSHI_DST = "/kaggle/working/moshi_repo"

if not os.path.exists(MOSHI_DST):
    print(f"Copying moshi repo to {MOSHI_DST} …")
    ret = subprocess.run(["cp", "-r", MOSHI_SRC, MOSHI_DST], capture_output=True, text=True)
    if ret.returncode != 0:
        raise RuntimeError(f"cp failed:\n{ret.stderr}")
    print("Copy done")
else:
    print(f"Repo already at {MOSHI_DST}, skipping copy")

# Now install editable from the writable copy.
print("Installing moshi (editable) …")
ret = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", MOSHI_DST],
    capture_output=True, text=True
)
if ret.returncode != 0:
    # Print full pip output so we can diagnose any further issues.
    print("pip stdout:", ret.stdout)
    print("pip stderr:", ret.stderr)
    raise RuntimeError("moshi editable install failed — see output above")
print("moshi installed (editable)")

# pip install -e writes a .pth file that only takes effect on interpreter
# startup. Since we're mid-session we must manually add the path now.
import site, importlib, sys
site.addsitedir(site.getsitepackages()[0])  # reload .pth files
# Also add the source root directly as a fallback.
MOSHI_SRC_ROOT = MOSHI_DST  # /kaggle/working/moshi_repo
if MOSHI_SRC_ROOT not in sys.path:
    sys.path.insert(0, MOSHI_SRC_ROOT)
importlib.invalidate_caches()

# Confirm moshi is now importable before proceeding.
try:
    import moshi
    print(f"moshi importable from: {moshi.__file__}")
except ModuleNotFoundError as e:
    raise RuntimeError(
        f"moshi still not importable after sys.path fix: {e}\n"
        f"sys.path = {sys.path}"
    )

import transformers, bitsandbytes as bnb
print("transformers:", transformers.__version__)
print("bitsandbytes:", bnb.__version__)
print("=== installs OK ===")


In [3]:
# Cell 3b — Monkey-patch CUDAGraphed to a no-op.
# Must run AFTER moshi is installed (Cell 3) but BEFORE any import of
# moshi.models.lm_gen, which is where the CUDAGraphed wrapper is actually used.
#
# Why: Moshi wraps LMGen forward paths with CUDAGraphed for fast inference.
# CUDA graphs cannot coexist with gradient checkpointing — the first backward
# pass will silently corrupt the graph. This patch is safe to leave on for all
# training sessions; re-enable CUDAGraphed only for standalone inference/eval.

import moshi.utils.compile as _moshi_compile

class _NoGraph:
    def __init__(self, fn, *a, **kw): self.fn = fn
    def __call__(self, *a, **kw):    return self.fn(*a, **kw)

_moshi_compile.CUDAGraphed = _NoGraph
print("CUDAGraphed monkey-patched to no-op")

# Sanity: confirm torch.compile is still off (persists across cells via os.environ).
import torch
assert torch._dynamo.config.disable, "torch.compile should still be disabled"
print("torch.compile still disabled — both patches active")


CUDAGraphed monkey-patched to no-op
torch.compile still disabled — both patches active


## Cell 4 — Load teacher + Mimi (fp16)

In [4]:
import torch, pathlib, time, shutil
from moshi.models.loaders import CheckpointInfo

REPO_ID  = "kyutai/moshiko-pytorch-bf16"
LOAD_DIR = pathlib.Path("/tmp/moshiko-weights")   # /tmp, not /kaggle/working
LOAD_DIR.mkdir(parents=True, exist_ok=True)

MOSHI_FILE = "model.safetensors"
MIMI_FILE  = "tokenizer-e351c8d8-checkpoint125.safetensors"
TOK_FILE   = "tokenizer_spm_32k_3.model"
FILES = {
    MOSHI_FILE : 14_000_000_000,
    MIMI_FILE  : 350_000_000,
    TOK_FILE   : 500_000,
}

def already_done(filename, min_size):
    p = LOAD_DIR / filename
    return p.exists() and p.stat().st_size >= min_size

from huggingface_hub import hf_hub_download

all_ok = all(already_done(f, s) for f, s in FILES.items())
if all_ok:
    print("All files already in /tmp — skipping download")
else:
    for filename, min_size in FILES.items():
        if already_done(filename, min_size):
            print(f"SKIP {filename} (already complete)")
            continue
        attempt = 0
        while True:
            attempt += 1
            current = (LOAD_DIR / filename).stat().st_size if (LOAD_DIR / filename).exists() else 0
            print(f"[attempt {attempt}] {filename}  {current/1e9:.2f} / ~{min_size/1e9:.2f} GB")
            try:
                hf_hub_download(
                    repo_id   = REPO_ID,
                    filename  = filename,
                    local_dir = str(LOAD_DIR),
                    force_download = False,
                )
                if already_done(filename, min_size):
                    print(f"  OK")
                    break
            except Exception as e:
                print(f"  Error: {e} — retrying in 10s")
                time.sleep(10)

# Verify
moshi_weights = LOAD_DIR / MOSHI_FILE
mimi_weights  = LOAD_DIR / MIMI_FILE
tokenizer     = LOAD_DIR / TOK_FILE
for f in [moshi_weights, mimi_weights, tokenizer]:
    assert f.exists(), f"Missing: {f.name}"
    print(f"  {f.name:55s} {f.stat().st_size/1e9:.3f} GB")

info = CheckpointInfo(
    moshi_weights = moshi_weights,
    mimi_weights  = mimi_weights,
    tokenizer     = tokenizer,
    lm_config     = None,
)

# Load to CPU first — 15.4 GB fp16 won't fit on one T4
print("\nLoading teacher LM to CPU …")
teacher_lm = info.get_moshi(device="cpu", dtype=torch.float16)
print("Loading Mimi to CPU …")
mimi = info.get_mimi(device="cpu")

def count_params(m):
    return sum(p.numel() for p in m.parameters())

print(f"\nTeacher LM params     : {count_params(teacher_lm) / 1e9:.3f} B")
print(f"Teacher LM mem (fp16) : {count_params(teacher_lm) * 2 / 1e9:.2f} GB")
print(f"Mimi params           : {count_params(mimi) / 1e6:.1f} M")

# ── Delete weights from disk — no longer needed once loaded into RAM ──────────
print("\nDeleting weights from /tmp (loaded into RAM, disk copy not needed) …")
shutil.rmtree(LOAD_DIR)
print(f"Deleted {LOAD_DIR}")

stat = __import__('os').statvfs("/kaggle/working")
print(f"Disk free: {stat.f_bavail * stat.f_frsize / 1e9:.1f} GB")

# ── Shard teacher across both GPUs layer-by-layer ─────────────────────────────
print("\nClearing GPU memory before shard …")
torch.cuda.empty_cache()
for i in range(2):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  cuda:{i} before shard: free {free/1e9:.2f} / total {total/1e9:.2f} GB")

print("\nSharding teacher layer-by-layer …")
teacher_lm.emb.to("cuda:0")
teacher_lm.text_emb.to("cuda:0")

SPLIT = 16
for layer in teacher_lm.transformer.layers[:SPLIT]:
    layer.to("cuda:0")
print(f"  Layers  0-{SPLIT-1} → cuda:0")

for layer in teacher_lm.transformer.layers[SPLIT:]:
    layer.to("cuda:1")
print(f"  Layers {SPLIT}-31 → cuda:1")

for attr in ["out_norm", "text_linear",
             "depformer_in", "depformer",
             "depformer_emb", "depformer_text_emb", "linears"]:
    if hasattr(teacher_lm, attr):
        getattr(teacher_lm, attr).to("cuda:1")
print("  Heads + Depformer → cuda:1")

mimi = mimi.to(device="cuda:0", dtype=torch.float16)
print("  Mimi → cuda:0 (fp16)")

torch.cuda.synchronize()
for i in range(2):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i} after shard: free {free/1e9:.2f} / total {total/1e9:.2f} GB")

l0  = next(teacher_lm.transformer.layers[0].parameters()).device
l31 = next(teacher_lm.transformer.layers[31].parameters()).device
assert str(l0)  == "cuda:0", f"layer 0 on wrong device: {l0}"
assert str(l31) == "cuda:1", f"layer 31 on wrong device: {l31}"
print(f"\nTeacher layer  0 → {l0}")
print(f"Teacher layer 31 → {l31}")

_orig_layers = teacher_lm.transformer.layers
def _sharded_forward(x, *args, **kwargs):
    for layer in _orig_layers[:SPLIT]:
        x = layer(x)
    x = x.to("cuda:1", non_blocking=True)
    for layer in _orig_layers[SPLIT:]:
        x = layer(x)
    return x
teacher_lm.transformer.forward = _sharded_forward
print("Cross-GPU forward patch applied")

print("\nWarming up teacher KV cache …")
with torch.inference_mode():
    dummy = torch.zeros(1, teacher_lm.num_codebooks, 8,
                        dtype=torch.long, device="cuda:0")
    _ = teacher_lm.forward_text(dummy)
print("Teacher warmup OK")
print("\n=== Cell 4 PASSED ===")


[attempt 1] model.safetensors  0.00 / ~14.00 GB
  OK
[attempt 1] tokenizer-e351c8d8-checkpoint125.safetensors  0.00 / ~0.35 GB
  OK
[attempt 1] tokenizer_spm_32k_3.model  0.00 / ~0.00 GB
  OK
  model.safetensors                                       15.376 GB
  tokenizer-e351c8d8-checkpoint125.safetensors            0.385 GB
  tokenizer_spm_32k_3.model                               0.001 GB

Loading teacher LM to CPU …
Loading Mimi to CPU …

Teacher LM params     : 7.688 B
Teacher LM mem (fp16) : 15.38 GB
Mimi params           : 79.3 M

Deleting weights from /tmp (loaded into RAM, disk copy not needed) …
Deleted /tmp/moshiko-weights
Disk free: 17.7 GB

Clearing GPU memory before shard …
  cuda:0 before shard: free 15.53 / total 15.64 GB
  cuda:1 before shard: free 15.53 / total 15.64 GB

Sharding teacher layer-by-layer …
  Layers  0-15 → cuda:0
  Layers 16-31 → cuda:1
  Heads + Depformer → cuda:1
  Mimi → cuda:0 (fp16)
cuda:0 after shard: free 8.21 / total 15.64 GB
cuda:1 after shard: 

## Cell 4b — Mirror HuggingFace weights to Kaggle Dataset

> **Run this once, right after Cell 4 finishes downloading.**  
> The HuggingFace cache at `~/.cache/huggingface` is ephemeral — wiped every session.  
> Mirroring to `/kaggle/working` lets you version it as a Kaggle Dataset and load  
> locally in all future sessions, saving ~20 min of re-downloading per session.


In [ ]:
import pathlib, subprocess

# Weights are already at /kaggle/working/moshiko-weights from Cell 4.
# Just report sizes and remind you to push the Dataset.

WORKING = pathlib.Path("/kaggle/working")

print("Files in /kaggle/working:")
for p in sorted(WORKING.iterdir()):
    if p.is_dir():
        sz = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e9
        print(f"  {p.name:<35} {sz:6.1f} GB")
    else:
        print(f"  {p.name:<35} {p.stat().st_size/1e6:6.1f} MB")

print()
print("Weights are already in /kaggle/working/moshiko-weights (downloaded by Cell 4).")
print("Push them as a Kaggle Dataset so future sessions skip the download entirely:")
print()
print("  First time:")
print("    kaggle datasets create -p /kaggle/working/moshiko-weights --dir-mode zip")
print()
print("  Subsequent sessions — load locally:")
print("    info = CheckpointInfo.from_pretrained('/kaggle/input/moshiko-weights/moshiko-weights')")


## Cell 5 — ~~Shard teacher to cuda:1~~ (folded into Cell 4)

In [ ]:
# Shard is now done inside Cell 4 (layer-by-layer across both GPUs).
# Nothing to do here.
import torch
torch.cuda.synchronize()
for i in range(2):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i}: free {free/1e9:.2f} / total {total/1e9:.2f} GB")


## Cell 6 — Pre-implementation checklist
> Every assertion must pass. Hard stop if any fails.

In [5]:
import torch, transformers, bitsandbytes as bnb

assert torch.cuda.device_count() == 2
# Hardware bf16 check — CC >= 8.0 means Ampere+; T4 is sm_75 so this must be False.
assert not any(
    torch.cuda.get_device_capability(i)[0] >= 8
    for i in range(torch.cuda.device_count())
), (f"Unexpected bf16-capable GPU detected. Expected T4 (sm_75), "
    f"got capabilities: {[torch.cuda.get_device_capability(i) for i in range(torch.cuda.device_count())]}")
assert transformers.__version__ >= "4.44.0", transformers.__version__

# Teacher is sharded: layers 0-15 on cuda:0, layers 16-31 + heads on cuda:1
assert teacher_lm is not None and mimi is not None
l0_device = next(teacher_lm.transformer.layers[0].parameters()).device
assert l0_device.type == "cuda", f"teacher must be on CUDA, got {l0_device}"

# Teacher forward shape check
with torch.inference_mode():
    dummy_codes = torch.zeros(
        1, teacher_lm.num_codebooks, 8, dtype=torch.long, device="cuda:0")  # emb lives on cuda:0
    h_t, text_logits_t = teacher_lm.forward_text(dummy_codes)

print("teacher hidden      :", h_t.shape)           # [1, 8, 4096]
print("teacher text_logits :", text_logits_t.shape) # [1, 1, 8, 32000]
assert h_t.shape[-1]          == 4096,  f"expected teacher_dim=4096, got {h_t.shape[-1]}"
assert text_logits_t.shape[-1] == 32000, f"expected vocab=32000, got {text_logits_t.shape[-1]}"

# num_codebooks must be 17 (1 text + 16 audio)
assert teacher_lm.num_codebooks == 17, (
    f"expected 17 codebooks, got {teacher_lm.num_codebooks}")

# 8-bit optimizer importable
_ = bnb.optim.PagedAdamW8bit
print("PagedAdamW8bit importable OK")

print("=== pre-implementation checklist PASSED ===")


teacher hidden      : torch.Size([1, 8, 4096])
teacher text_logits : torch.Size([1, 1, 8, 32000])
PagedAdamW8bit importable OK
=== pre-implementation checklist PASSED ===


## Cell 7 — Write `smol_temporal.py` to the Moshi repo

This is the only new file we implement in S0. It is the complete `SmolTemporalTransformer` — a drop-in replacement for Moshi's 32-layer Helium TT.

In [6]:
import pathlib, base64, importlib, sys, torch

# smol_temporal.py is embedded as base64 to avoid any string quoting issues.
_SRC_B64 = "IyBtb3NoaS9tb2RlbHMvc21vbF90ZW1wb3JhbC5weQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcywgZmllbGQKZnJvbSB0eXBpbmcgaW1wb3J0IE9wdGlvbmFsCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IHRyYW5zZm9ybWVycwpmcm9tIC4ubW9kdWxlcy5zdHJlYW1pbmcgaW1wb3J0IFN0cmVhbWluZ01vZHVsZSwgU3RhdGUKCgpAZGF0YWNsYXNzCmNsYXNzIF9TbW9sU3RhdGUoU3RhdGUpOgogICAgIyBTdGF0ZS5iYXRjaF9zaXplIGFuZCBTdGF0ZS5kZXZpY2UgYXJlIGluaGVyaXRlZCAocmVxdWlyZWQsIHBvc2l0aW9uYWwpLgogICAgIyBwYXN0X2tleV92YWx1ZXMgaGFzIGEgZGVmYXVsdCBzbyBpdCBjb21lcyBhZnRlciB0aGVtLgogICAgcGFzdF9rZXlfdmFsdWVzOiBPcHRpb25hbFt0dXBsZV0gPSBmaWVsZChkZWZhdWx0PU5vbmUpCgogICAgZGVmIHJlc2V0KHNlbGYsIHJlc2V0X21hc2s6IHRvcmNoLlRlbnNvcikgLT4gTm9uZToKICAgICAgICBzdXBlcigpLnJlc2V0KHJlc2V0X21hc2spCiAgICAgICAgc2VsZi5wYXN0X2tleV92YWx1ZXMgPSBOb25lCgoKY2xhc3MgU21vbFRlbXBvcmFsVHJhbnNmb3JtZXIoU3RyZWFtaW5nTW9kdWxlW19TbW9sU3RhdGVdKToKICAgICIiIkRyb3AtaW4gcmVwbGFjZW1lbnQgZm9yIE1vc2hpJ3MgMzItbGF5ZXIgSGVsaXVtIFRlbXBvcmFsIFRyYW5zZm9ybWVyLgoKICAgIElucHV0ICA6IFtCLCBULCB0ZWFjaGVyX2RpbT00MDk2XQogICAgT3V0cHV0IDogW0IsIFQsIHRlYWNoZXJfZGltPTQwOTZdCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICB0ZWFjaGVyX2RpbTogaW50ID0gNDA5NiwKICAgICAgICBzdHVkZW50X2RpbTogaW50ID0gMjA0OCwKICAgICAgICBoZl9uYW1lOiBzdHIgPSAiSHVnZ2luZ0ZhY2VUQi9TbW9sTE0yLTEuN0IiLAogICAgICAgIHJvcGVfdGhldGE6IGZsb2F0ID0gMTBfMDAwLjAsCiAgICAgICAgZGV2aWNlOiBzdHIgPSAiY3VkYToxIiwKICAgICAgICBkdHlwZTogdG9yY2guZHR5cGUgPSB0b3JjaC5mbG9hdDE2LAogICAgKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLnRlYWNoZXJfZGltID0gdGVhY2hlcl9kaW0KICAgICAgICBzZWxmLnN0dWRlbnRfZGltID0gc3R1ZGVudF9kaW0KCiAgICAgICAgY2ZnID0gdHJhbnNmb3JtZXJzLkF1dG9Db25maWcuZnJvbV9wcmV0cmFpbmVkKGhmX25hbWUpCiAgICAgICAgY2ZnLnJvcGVfdGhldGEgPSByb3BlX3RoZXRhCiAgICAgICAgY2ZnLnVzZV9jYWNoZSA9IFRydWUKICAgICAgICBjZmcuYXR0bl9pbXBsZW1lbnRhdGlvbiA9ICJlYWdlciIKICAgICAgICBzZWxmLmJhY2tib25lID0gdHJhbnNmb3JtZXJzLkF1dG9Nb2RlbC5mcm9tX3ByZXRyYWluZWQoCiAgICAgICAgICAgIGhmX25hbWUsIGNvbmZpZz1jZmcsIHRvcmNoX2R0eXBlPWR0eXBlLAogICAgICAgICkKICAgICAgICBpZiBoYXNhdHRyKHNlbGYuYmFja2JvbmUsICJlbWJlZF90b2tlbnMiKToKICAgICAgICAgICAgc2VsZi5iYWNrYm9uZS5lbWJlZF90b2tlbnMgPSBubi5JZGVudGl0eSgpCgogICAgICAgIHNlbGYuaW5fYWRhcHRlciAgPSBubi5MaW5lYXIodGVhY2hlcl9kaW0sIHN0dWRlbnRfZGltLCBiaWFzPUZhbHNlKQogICAgICAgIHNlbGYub3V0X2FkYXB0ZXIgPSBubi5MaW5lYXIoc3R1ZGVudF9kaW0sIHRlYWNoZXJfZGltLCBiaWFzPUZhbHNlKQogICAgICAgIG5uLmluaXQubm9ybWFsXyhzZWxmLmluX2FkYXB0ZXIud2VpZ2h0LCAgc3RkPTEuMCAvICh0ZWFjaGVyX2RpbSAqKiAwLjUpKQogICAgICAgIG5uLmluaXQubm9ybWFsXyhzZWxmLm91dF9hZGFwdGVyLndlaWdodCwgc3RkPTEuMCAvIChzdHVkZW50X2RpbSAqKiAwLjUpKQoKICAgICAgICBzZWxmLnRvKGRldmljZT1kZXZpY2UsIGR0eXBlPWR0eXBlKQoKICAgIGRlZiBfaW5pdF9zdHJlYW1pbmdfc3RhdGUoc2VsZiwgYmF0Y2hfc2l6ZTogaW50KSAtPiBfU21vbFN0YXRlOgogICAgICAgIGRldmljZSA9IHNlbGYuaW5fYWRhcHRlci53ZWlnaHQuZGV2aWNlCiAgICAgICAgcmV0dXJuIF9TbW9sU3RhdGUoYmF0Y2hfc2l6ZT1iYXRjaF9zaXplLCBkZXZpY2U9ZGV2aWNlLCBwYXN0X2tleV92YWx1ZXM9Tm9uZSkKCiAgICBkZWYgZm9yd2FyZCgKICAgICAgICBzZWxmLAogICAgICAgIHg6IHRvcmNoLlRlbnNvciwKICAgICAgICBjcm9zc19hdHRlbnRpb25fc3JjOiBPcHRpb25hbFt0b3JjaC5UZW5zb3JdID0gTm9uZSwKICAgICkgLT4gdG9yY2guVGVuc29yOgogICAgICAgIGFzc2VydCBjcm9zc19hdHRlbnRpb25fc3JjIGlzIE5vbmUsICgKICAgICAgICAgICAgIlNtb2xUZW1wb3JhbFRyYW5zZm9ybWVyOiBjcm9zcy1hdHRlbnRpb24gbm90IHN1cHBvcnRlZCIpCiAgICAgICAgYXNzZXJ0IHguZGltKCkgPT0gMyBhbmQgeC5zaGFwZVstMV0gPT0gc2VsZi50ZWFjaGVyX2RpbSwgKAogICAgICAgICAgICBmImV4cGVjdGVkIFtCLCBULCB7c2VsZi50ZWFjaGVyX2RpbX1dLCBnb3Qge3R1cGxlKHguc2hhcGUpfSIpCgogICAgICAgICMgTW92ZSBpbnB1dCB0byBzdHVkZW50IGRldmljZSAoZW1iIHN1bSBpcyBvbiBjdWRhOjAsIGJhY2tib25lIG9uIGN1ZGE6MSkKICAgICAgICB4ID0geC50byhzZWxmLmluX2FkYXB0ZXIud2VpZ2h0LmRldmljZSkKICAgICAgICBoID0gc2VsZi5pbl9hZGFwdGVyKHgpCgogICAgICAgIHBhc3Rfa3YgPSAoc2VsZi5fc3RyZWFtaW5nX3N0YXRlLnBhc3Rfa2V5X3ZhbHVlcwogICAgICAgICAgICAgICAgICAgaWYgc2VsZi5fc3RyZWFtaW5nX3N0YXRlIGlzIG5vdCBOb25lIGVsc2UgTm9uZSkKCiAgICAgICAgb3V0ID0gc2VsZi5iYWNrYm9uZSgKICAgICAgICAgICAgaW5wdXRzX2VtYmVkcz1oLAogICAgICAgICAgICBwYXN0X2tleV92YWx1ZXM9cGFzdF9rdiwKICAgICAgICAgICAgdXNlX2NhY2hlPShzZWxmLl9zdHJlYW1pbmdfc3RhdGUgaXMgbm90IE5vbmUpLAogICAgICAgICAgICByZXR1cm5fZGljdD1UcnVlLAogICAgICAgICkKCiAgICAgICAgaWYgc2VsZi5fc3RyZWFtaW5nX3N0YXRlIGlzIG5vdCBOb25lOgogICAgICAgICAgICBzZWxmLl9zdHJlYW1pbmdfc3RhdGUucGFzdF9rZXlfdmFsdWVzID0gb3V0LnBhc3Rfa2V5X3ZhbHVlcwoKICAgICAgICByZXR1cm4gc2VsZi5vdXRfYWRhcHRlcihvdXQubGFzdF9oaWRkZW5fc3RhdGUpCgogICAgZGVmIHN0dWRlbnRfc3RhdGVfZGljdChzZWxmKToKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAiYmFja2JvbmUiOiAgICBzZWxmLmJhY2tib25lLnN0YXRlX2RpY3QoKSwKICAgICAgICAgICAgImluX2FkYXB0ZXIiOiAgc2VsZi5pbl9hZGFwdGVyLnN0YXRlX2RpY3QoKSwKICAgICAgICAgICAgIm91dF9hZGFwdGVyIjogc2VsZi5vdXRfYWRhcHRlci5zdGF0ZV9kaWN0KCksCiAgICAgICAgfQoKICAgIGRlZiBsb2FkX3N0dWRlbnRfc3RhdGVfZGljdChzZWxmLCBzZDogZGljdCk6CiAgICAgICAgc2VsZi5iYWNrYm9uZS5sb2FkX3N0YXRlX2RpY3Qoc2RbImJhY2tib25lIl0pCiAgICAgICAgc2VsZi5pbl9hZGFwdGVyLmxvYWRfc3RhdGVfZGljdChzZFsiaW5fYWRhcHRlciJdKQogICAgICAgIHNlbGYub3V0X2FkYXB0ZXIubG9hZF9zdGF0ZV9kaWN0KHNkWyJvdXRfYWRhcHRlciJdKQo="

out_path = pathlib.Path("/kaggle/working/moshi_repo/moshi/models/smol_temporal.py")
out_path.write_bytes(base64.b64decode(_SRC_B64))
print(f"Written : {out_path}")
print(f"Size    : {out_path.stat().st_size} bytes")

# Invalidate module cache and re-import
for key in list(sys.modules.keys()):
    if "smol_temporal" in key:
        del sys.modules[key]
importlib.invalidate_caches()

from moshi.models.smol_temporal import SmolTemporalTransformer, _SmolState
print("Import OK:", SmolTemporalTransformer)

s = _SmolState(batch_size=1, device=torch.device("cpu"), past_key_values=None)
print(f"_SmolState OK: batch_size={s.batch_size}, device={s.device}, past_key_values={s.past_key_values}")
print("=== Cell 7 PASSED ===")


Written : /kaggle/working/moshi_repo/moshi/models/smol_temporal.py
Size    : 3680 bytes
Import OK: <class 'moshi.models.smol_temporal.SmolTemporalTransformer'>
_SmolState OK: batch_size=1, device=cpu, past_key_values=None
=== Cell 7 PASSED ===


## Cell 8 — Build student shell via `replace_temporal_transformer`

Load a fresh Moshi instance as the student shell, swap its 32-layer Helium TT for `SmolTemporalTransformer`, and freeze everything except the student backbone + adapters.

In [7]:
import torch, pathlib
from moshi.models.smol_temporal import SmolTemporalTransformer


def replace_temporal_transformer(
    lm_model,
    hf_name: str = "HuggingFaceTB/SmolLM2-1.7B",
    device: str = "cuda:1",       # student lives on cuda:1
    dtype: torch.dtype = torch.float16,
) -> SmolTemporalTransformer:
    teacher_dim = getattr(lm_model, "transformer_dim", 4096)
    new_tt = SmolTemporalTransformer(
        teacher_dim=teacher_dim,
        student_dim=2048,
        hf_name=hf_name,
        rope_theta=10_000.0,
        device=device,
        dtype=dtype,
    )
    old = lm_model.transformer
    lm_model.transformer = new_tt
    del old
    torch.cuda.empty_cache()

    for name, p in lm_model.named_parameters():
        if name.startswith("transformer."):
            p.requires_grad_(True)
        else:
            p.requires_grad_(False)
    return new_tt


# ── Memory layout ──────────────────────────────────────────────────────────────
# cuda:0 : teacher layers  0-15           (~7.7 GB)
# cuda:1 : teacher layers 16-31 + heads   (~8.6 GB)
#        + student SmolLM2 backbone        (~3.4 GB)
#        + frozen Moshi heads on student   (~0.5 GB)
#        ─────────────────────────────────────────────
#          total cuda:1                   ~12.5 GB  (3.1 GB headroom for activations)

# Reuse the CheckpointInfo already constructed in Cell 4.
# The weights were deleted from disk after loading into RAM,
# so we cannot reconstruct CheckpointInfo from file paths.
# info is still in scope from Cell 4.
print("Loading student Moshi shell to CPU (reusing Cell 4 CheckpointInfo) …")
# load_weight=False — skip loading 15.4 GB weights, just build the architecture.
# replace_temporal_transformer immediately swaps out the Helium TT anyway,
# so we only need the frozen heads (emb, out_norm, text_linear, depformer)
# which are initialized from the architecture config, not the weights file.
student_lm = info.get_moshi(device="cpu", dtype=torch.float16, load_weight=False)
print(f"Student shell on CPU OK")

# Swap Helium TT → SmolLM2 while still on CPU
print("Replacing Helium TT with SmolLM2-1.7B (on CPU) …")
new_tt = replace_temporal_transformer(
    student_lm,
    hf_name="HuggingFaceTB/SmolLM2-1.7B",
    device="cpu",                  # stay on CPU for now
    dtype=torch.float16,
)
torch.cuda.empty_cache()
for i in range(2):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  cuda:{i} after TT swap (CPU): free {free/1e9:.2f} / {total/1e9:.2f} GB")

# Move student transformer → cuda:1
print("\nMoving student transformer → cuda:1 …")
student_lm.transformer.to("cuda:1")

# Move frozen Moshi heads → cuda:1
# (these are used by forward_text after the transformer output)
print("Moving frozen Moshi heads → cuda:1 …")
for attr in ["out_norm", "text_linear",
             "depformer_in", "depformer",
             "depformer_emb", "depformer_text_emb", "linears"]:
    if hasattr(student_lm, attr):
        getattr(student_lm, attr).to("cuda:1")

# Embeddings must match where forward_text sends input —
# teacher emb is on cuda:0, student emb must also be on cuda:0
# because the student's forward_text still starts with embedding lookup.
print("Moving student emb → cuda:0 …")
if hasattr(student_lm, "emb"):
    student_lm.emb.to("cuda:0")
if hasattr(student_lm, "text_emb"):
    student_lm.text_emb.to("cuda:0")

torch.cuda.synchronize()
for i in range(2):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i} final: free {free/1e9:.2f} / {total/1e9:.2f} GB")

n_train = sum(p.numel() for p in student_lm.parameters() if p.requires_grad)
n_froz  = sum(p.numel() for p in student_lm.parameters() if not p.requires_grad)
print(f"\nTrainable : {n_train/1e9:.3f} B  (expected ~1.7 B)")
print(f"Frozen    : {n_froz/1e9:.3f} B")
print("\n=== Cell 8 PASSED ===")


Loading student Moshi shell to CPU (reusing Cell 4 CheckpointInfo) …
Student shell on CPU OK
Replacing Helium TT with SmolLM2-1.7B (on CPU) …
  cuda:0 after TT swap (CPU): free 8.16 / 15.64 GB
  cuda:1 after TT swap (CPU): free 7.08 / 15.64 GB

Moving student transformer → cuda:1 …
Moving frozen Moshi heads → cuda:1 …
Moving student emb → cuda:0 …
cuda:0 final: free 7.60 / 15.64 GB
cuda:1 final: free 1.86 / 15.64 GB

Trainable : 1.627 B  (expected ~1.7 B)
Frozen    : 1.111 B

=== Cell 8 PASSED ===


## Cell 9 — Integration sanity checks
> All assertions must pass. See §4 of `MoshiCompressionExecution.md` for the failure-mode table.

In [8]:
import torch
from moshi.modules.streaming import StreamingModule


def sanity_check_integration(student_lm, batch_size: int = 2, seq_len: int = 16):
    student_lm.eval()
    device = "cuda:0"
    K = student_lm.num_codebooks

    # 1. num_codebooks
    assert K == 17, f"expected 17 streams, got {K}"

    # 2. forward_text output shapes
    fake = torch.zeros(batch_size, K, seq_len, dtype=torch.long, device=device)
    fake[:, 0] = 1   # text stream: valid (non-pad) token id

    with torch.inference_mode():
        transformer_out, text_logits = student_lm.forward_text(fake)

    print("transformer_out :", tuple(transformer_out.shape))
    print("text_logits     :", tuple(text_logits.shape))
    assert transformer_out.shape == (batch_size, seq_len, 4096), (
        f"expected [{batch_size},{seq_len},4096], got {tuple(transformer_out.shape)}")
    assert text_logits.shape == (batch_size, 1, seq_len, 32000), (
        f"expected [{batch_size},1,{seq_len},32000], got {tuple(text_logits.shape)}")

    # 3. Depformer projection seeds (one per codebook)
    dep_dim = student_lm.depformer_in[0].out_features
    assert dep_dim == 1024, f"expected depformer_dim=1024, got {dep_dim}"
    for k in range(student_lm.dep_q):
        seed_k = student_lm.depformer_in[k](transformer_out[:, -1:])
        assert seed_k.shape == (batch_size, 1, 1024), (
            f"depformer seed {k}: expected [{batch_size},1,1024], got {tuple(seed_k.shape)}")
    print(f"{student_lm.dep_q} depformer seeds [B,1,1024] OK")

    # 4. No NaN / Inf
    assert torch.isfinite(transformer_out).all(), "NaN/Inf in transformer_out"
    assert torch.isfinite(text_logits).all(),     "NaN/Inf in text_logits"

    # 5. Activation magnitude (RMSNorm output should be O(1))
    mean_abs = transformer_out.abs().mean().item()
    max_abs  = transformer_out.abs().max().item()
    print(f"transformer_out |x|: mean={mean_abs:.3f}  max={max_abs:.3f}")
    assert 0.01 < mean_abs < 50.0, (
        f"Magnitude out of range (mean={mean_abs:.4f}). "
        "Check adapter init std or fp16 saturation.")

    # 6. StreamingContainer round-trip (past_key_values compose across frames)
    assert isinstance(student_lm.transformer, StreamingModule), (
        "student_lm.transformer must be a StreamingModule")

    frame = torch.zeros(batch_size, K, 1, dtype=torch.long, device=device)
    frame[:, 0] = 1
    with student_lm.transformer.streaming(batch_size=batch_size):
        for t in range(4):
            with torch.inference_mode():
                tfo, tl = student_lm.forward_text(frame)
            assert tfo.shape == (batch_size, 1, 4096), (
                f"streaming frame {t}: got {tuple(tfo.shape)}")
            assert tl.shape  == (batch_size, 1, 1, 32000), (
                f"streaming frame {t}: got {tuple(tl.shape)}")
            assert torch.isfinite(tfo).all(), f"NaN in streaming frame {t}"
    print("streaming round-trip OK (4 frames, past_key_values accumulating)")

    # 7. Trainable / frozen counts
    n_trainable = sum(p.numel() for p in student_lm.parameters() if p.requires_grad)
    n_frozen    = sum(p.numel() for p in student_lm.parameters() if not p.requires_grad)
    print(f"trainable : {n_trainable/1e9:.3f} B  (expected ~1.7 B)")
    print(f"frozen    : {n_frozen/1e9:.3f} B    (expected ~5.9–6.0 B)")
    assert 1.5e9 < n_trainable < 2.0e9, (
        f"Unexpected trainable count: {n_trainable/1e9:.3f} B. "
        "Check requires_grad freeze loop.")

    print("=== integration sanity checks PASSED ===")


sanity_check_integration(student_lm)


We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


transformer_out : (2, 16, 4096)
text_logits     : (2, 1, 16, 32000)
8 depformer seeds [B,1,1024] OK
transformer_out |x|: mean=0.798  max=4.043
streaming round-trip OK (4 frames, past_key_values accumulating)
trainable : 1.627 B  (expected ~1.7 B)
frozen    : 1.111 B    (expected ~5.9–6.0 B)
=== integration sanity checks PASSED ===


## Cell 10 — 10 fake training steps

Confirms on random tensors that:
- Backward pass completes without OOM
- `GradScaler` is alive (no overflow on step 1)
- Gradient norms are finite
- 8-bit PagedAdamW step succeeds

> No real data is loaded. This is purely a compute-graph and memory-budget check.


In [9]:
import torch
import torch.nn.functional as F
import bitsandbytes as bnb
import time, os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Offload teacher to CPU — Phase-1 is teacher-free (cached targets)
print("Offloading teacher to CPU …")
teacher_lm.to("cpu")
torch.cuda.empty_cache()
torch.cuda.synchronize()
for i in range(2):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  cuda:{i} after offload: free {free/1e9:.2f} / {total/1e9:.2f} GB")

# ── Move trainable student params to cuda:0 ───────────────────────────────────
# cuda:0 has ~14 GB free after teacher offload.
# We move the SmolLM2 backbone + adapters there for training.
# Frozen heads (out_norm, text_linear, depformer) stay on cuda:1.
# SmolTemporalTransformer.forward already has x.to(self.in_adapter.weight.device)
# so it will pick up the new device automatically.
print("\nMoving trainable student components → cuda:0 …")
student_lm.transformer.to("cuda:0")

# Move out_norm and text_linear to cuda:0 — they consume transformer_out directly
for attr in ["out_norm", "text_linear"]:
    if hasattr(student_lm, attr):
        getattr(student_lm, attr).to("cuda:0")
print("out_norm + text_linear → cuda:0")

# depformer_in also needs to be on cuda:0 (takes transformer_out as input)
if hasattr(student_lm, "depformer_in"):
    student_lm.depformer_in.to("cuda:0")
print("depformer_in → cuda:0")

# Cast trainable params to fp32 (required for GradScaler.unscale_)
print("Casting trainable params to fp32 …")
for p in student_lm.parameters():
    if p.requires_grad:
        p.data = p.data.float()

torch.cuda.synchronize()
for i in range(2):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  cuda:{i} after move+cast: free {free/1e9:.2f} / {total/1e9:.2f} GB")

student_lm.train()
device = "cuda:0"
K = student_lm.num_codebooks
T = 64
B = 1

trainable_params = [p for p in student_lm.parameters() if p.requires_grad]
assert trainable_params

optimizer = bnb.optim.PagedAdamW8bit(trainable_params, lr=1e-4, weight_decay=0.01)
scaler    = torch.amp.GradScaler("cuda", init_scale=2**8)  # low init scale avoids step-1 overflow

backbone = student_lm.transformer.backbone
if hasattr(backbone, "gradient_checkpointing_enable"):
    backbone.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False})
    print("Gradient checkpointing enabled")

print(f"\nRunning 10 fake training steps  (B={B}, T={T}, K={K})")
print(f"Trainable: {sum(p.numel() for p in trainable_params)/1e9:.3f} B params")
print()

t0 = time.time()
for step in range(10):
    optimizer.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()

    fake_codes = torch.zeros(B, K, T, dtype=torch.long, device=device)
    fake_codes[:, 0] = 1

    with torch.amp.autocast("cuda", dtype=torch.float16):
        transformer_out, text_logits = student_lm.forward_text(fake_codes)
        target_hidden  = torch.randn_like(transformer_out)
        loss_hidden    = F.mse_loss(transformer_out, target_hidden)
        text_logits_2d = text_logits[:, 0].reshape(B * T, -1)
        text_labels    = torch.randint(0, 32000, (B * T,), device=text_logits_2d.device)
        loss_text      = F.cross_entropy(text_logits_2d, text_labels)
        loss = loss_hidden + 0.1 * loss_text

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
    scaler.step(optimizer)
    scaler.update()

    assert torch.isfinite(loss), f"step {step+1}: loss non-finite"

    # grad_norm may be inf on overflow steps (scaler skips optimizer.step) — just report
    gn_val = float(grad_norm)
    gn_str = f"{gn_val:.3f}" if torch.isfinite(torch.tensor(gn_val)) else "inf(overflow)"

    print(
        f"step {step+1:02d}  loss={loss.item():.4f} "
        f"(hid={loss_hidden.item():.4f} txt={loss_text.item():.4f})  "
        f"grad_norm={gn_str}  scale={scaler.get_scale():.0f}  "
        f"t={time.time()-t0:.1f}s"
    )

print()
torch.cuda.synchronize()
for i in range(2):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i} after backward: free {free/1e9:.2f} / {total/1e9:.2f} GB")

# Verify at least some steps had finite grad norms
print("\n=== 10 fake training steps PASSED ===")
print("Phase-1: student-only training (no teacher in VRAM, cached targets from disk).")


Offloading teacher to CPU …
  cuda:0 after offload: free 14.19 / 15.64 GB
  cuda:1 after offload: free 10.07 / 15.64 GB

Moving trainable student components → cuda:0 …
out_norm + text_linear → cuda:0
depformer_in → cuda:0
Casting trainable params to fp32 …
  cuda:0 after move+cast: free 6.23 / 15.64 GB
  cuda:1 after move+cast: free 10.07 / 15.64 GB
Gradient checkpointing enabled

Running 10 fake training steps  (B=1, T=64, K=17)
Trainable: 1.627 B params

step 01  loss=3.0809 (hid=2.0012 txt=10.7971)  grad_norm=7.621  scale=256  t=4.3s
step 02  loss=3.1127 (hid=2.0038 txt=11.0885)  grad_norm=2.293  scale=256  t=6.4s
step 03  loss=3.0794 (hid=2.0021 txt=10.7726)  grad_norm=2.630  scale=256  t=7.9s
step 04  loss=3.1089 (hid=1.9993 txt=11.0958)  grad_norm=1.085  scale=256  t=9.4s
step 05  loss=3.0710 (hid=1.9946 txt=10.7634)  grad_norm=1.574  scale=256  t=10.8s
step 06  loss=3.0838 (hid=1.9997 txt=10.8416)  grad_norm=0.824  scale=256  t=12.3s
step 07  loss=3.0819 (hid=1.9998 txt=10.8212)

## Cell 11 — Save checkpoint + MANIFEST.md + env.txt

In [10]:
import torch, os, json, random, subprocess, time, pathlib, shutil
import numpy as np
import transformers

CKPT_DIR = pathlib.Path("/kaggle/working")

# ── Free disk space by deleting weights (already loaded into RAM) ─────────────
weights_dir = CKPT_DIR / "moshiko-weights"
if weights_dir.exists():
    size = sum(f.stat().st_size for f in weights_dir.rglob("*") if f.is_file()) / 1e9
    shutil.rmtree(weights_dir)
    print(f"Deleted {weights_dir} ({size:.1f} GB freed)")
else:
    print("moshiko-weights already absent")

stat = os.statvfs("/kaggle/working")
free_gb = stat.f_bavail * stat.f_frsize / 1e9
print(f"Disk free after cleanup: {free_gb:.1f} GB")

# ── Checkpoint (fp16 weights only — optimizer state not needed for S0) ────────
def state_dict_fp16(module):
    # Move to CPU first — state_dict() on a GPU module allocates new GPU tensors
    return {k: v.cpu().half() if v.is_floating_point() else v.cpu()
            for k, v in module.state_dict().items()}

ckpt = {
    "step"        : 10,
    "phase"       : "S0_smoke",
    "wall_seconds": time.time(),

    "student_backbone": state_dict_fp16(student_lm.transformer.backbone),
    "in_adapter"      : state_dict_fp16(student_lm.transformer.in_adapter),
    "out_adapter"     : state_dict_fp16(student_lm.transformer.out_adapter),

    "torch_rng" : torch.get_rng_state(),
    "cuda_rng"  : torch.cuda.get_rng_state_all(),
    "numpy_rng" : np.random.get_state(),
    "python_rng": random.getstate(),

    "cache_shard_idx" : 0,
    "cache_sample_idx": 0,

    "torch_version"       : torch.__version__,
    "transformers_version": transformers.__version__,
    "notes": "S0 smoke test — 10 fake steps, fp16 weights, no optimizer state",
}

tmp_path   = CKPT_DIR / "ckpt.pt.tmp"
final_path = CKPT_DIR / "ckpt_step_10.pt"
print("\nSaving checkpoint …")
torch.save(ckpt, tmp_path)
os.replace(tmp_path, final_path)
size_gb = final_path.stat().st_size / 1e9
print(f"Checkpoint: {final_path}  ({size_gb:.2f} GB)")

# ── env.txt ──
env_out  = subprocess.run(["pip", "freeze"],    capture_output=True, text=True).stdout
nvid_out = subprocess.run(["nvidia-smi", "-q"], capture_output=True, text=True).stdout
(CKPT_DIR / "env.txt").write_text(
    "=== pip freeze ===\n" + env_out + "\n=== nvidia-smi -q ===\n" + nvid_out)

# ── train_log.jsonl ──
with open(CKPT_DIR / "train_log.jsonl", "a") as f:
    f.write(json.dumps({"step": 10, "phase": "S0_smoke",
                        "notes": "smoke test, random targets"}) + "\n")

# ── MANIFEST.md ──
(CKPT_DIR / "MANIFEST.md").write_text(f"""# MANIFEST — Session S0 (Smoke Test)

| Key | Value |
|---|---|
| phase | S0_smoke |
| step | 10 |
| torch | {torch.__version__} |
| transformers | {transformers.__version__} |
| checkpoint | ckpt_step_10.pt ({size_gb:.2f} GB) |
| notes | 10 fake steps, all assertions passed |

## Assertions passed
- Dual T4, sm_75, bf16=False
- Pre-implementation checklist
- Integration sanity checks (shapes, NaN, streaming, param counts)
- 10 fake backward passes without OOM

## Memory layout (Phase-1)
- Teacher offloaded to CPU; student-only training on cuda:0
- Phase-1 real training: teacher never loaded, cached targets from disk

## Next: S1 — Phase 0 teacher caching
""")

print("env.txt, train_log.jsonl, MANIFEST.md written")
print()
stat2 = os.statvfs("/kaggle/working")
print(f"Disk free after save: {stat2.f_bavail * stat2.f_frsize / 1e9:.1f} GB")
print()
print("Files in /kaggle/working:")
for p in sorted(CKPT_DIR.iterdir()):
    if p.is_file():
        print(f"  {p.name:<40} {p.stat().st_size/1e6:8.1f} MB")

print("\n=== S0 COMPLETE ===")


moshiko-weights already absent
Disk free after cleanup: 17.7 GB

Saving checkpoint …
Checkpoint: /kaggle/working/ckpt_step_10.pt  (3.26 GB)
env.txt, train_log.jsonl, MANIFEST.md written

Disk free after save: 17.7 GB

Files in /kaggle/working:
  MANIFEST.md                                   0.0 MB
  ckpt_step_10.pt                            3255.1 MB
  dataset-metadata.json                         0.0 MB
  env.txt                                       0.0 MB
  train_log.jsonl                               0.0 MB

=== S0 COMPLETE ===


## Cell 12 — Push to Kaggle Dataset

Run as the **very last cell**. Never run mid-training (upload competes with GPU for bandwidth).

In [23]:
import subprocess, json, pathlib, os

CKPT_DIR = pathlib.Path("/kaggle/working")
username = os.environ.get("KAGGLE_USERNAME", "tasfiatanha")

# Write metadata
metadata = {
    "title": "moshi-compression-smoke",
    "id": f"{username}/moshi-compression-smoke",
    "licenses": [{"name": "CC0-1.0"}]
}
(CKPT_DIR / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))
print(f"Dataset id: {username}/moshi-compression-smoke")

print("\nFiles to push:")
for p in sorted(CKPT_DIR.iterdir()):
    if p.is_file():
        print(f"  {p.name:<40} {p.stat().st_size/1e6:8.1f} MB")

# Always try create first — if dataset exists it returns 409/400, then we version.
print("\nRunning: kaggle datasets create …")
r = subprocess.run(
    ["kaggle", "datasets", "create", "-p", str(CKPT_DIR)],
    capture_output=True, text=True
)
print(r.stdout or "(no stdout)")

if r.returncode == 0:
    print(f"SUCCESS — kaggle.com/{username}/moshi-compression-smoke")
else:
    # Dataset already exists — bump version instead
    print(f"Create failed (rc={r.returncode}), trying version bump …")
    r2 = subprocess.run(
        ["kaggle", "datasets", "version",
         "-p", str(CKPT_DIR),
         "-m", "S0 smoke test complete"],
        capture_output=True, text=True
    )
    print(r2.stdout or "(no stdout)")
    if r2.returncode != 0:
        print("STDERR:", r2.stderr)
        print()
        print("Manual fallback — run this in a terminal:")
        print(f"  cd /kaggle/working && kaggle datasets create -p .")
    else:
        print(f"SUCCESS — kaggle.com/{username}/moshi-compression-smoke")


Dataset id: tasfiatanha/moshi-compression-smoke

Files to push:
  MANIFEST.md                                   0.0 MB
  ckpt_step_10.pt                            3255.1 MB
  dataset-metadata.json                         0.0 MB
  env.txt                                       0.0 MB
  train_log.jsonl                               0.0 MB

Running: kaggle datasets create …
Starting upload for file ckpt_step_10.pt
Error while trying to load upload info: KaggleObject.from_dict() got an unexpected keyword argument 'token'
Upload successful: ckpt_step_10.pt (3GB)
Starting upload for file env.txt
Error while trying to load upload info: KaggleObject.from_dict() got an unexpected keyword argument 'token'
Upload successful: env.txt (38KB)
Starting upload for file train_log.jsonl
Error while trying to load upload info: KaggleObject.from_dict() got an unexpected keyword argument 'token'
Upload successful: train_log.jsonl (73B)
Starting upload for file MANIFEST.md
Error while trying to load upload 